# Multi-Expert Natural Language Inference (NLI)
This notebook implements a hybrid NLI model combining a BERT-based semantic encoder with manual lexical and logic features using a Gating Network.

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score

## 1. Data Loading
Ensure your CSV files are located in `training_data/NLI/`.

In [4]:
train_path = "training_data/NLI/train.csv"
dev_path = "training_data/NLI/dev.csv"

train_df = pd.read_csv(train_path)
dev_df = pd.read_csv(dev_path)

train_dataset = Dataset.from_pandas(train_df)
dev_dataset = Dataset.from_pandas(dev_df)

print(f"Training samples: {len(train_dataset)}")
print(train_dataset)

Training samples: 24432
Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 24432
})


## 2. Feature Engineering & Preprocessing
We define manual features (negation, contrast, overlap) to supplement the transformer embeddings.

In [5]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

NEG_WORDS = {"not", "no", "never", "none", "nobody", "nothing", "neither", "nowhere", "hardly", "scarcely", "barely", "cannot", "can't", "won't", "isn't"}
CONTRAST_WORDS = {"but", "however", "although", "though", "yet", "despite"}

def extract_features(premise, hypothesis):
    p, h = premise.lower().split(), hypothesis.lower().split()
    p_set, h_set = set(p), set(h)

    overlap = len(p_set & h_set)
    union = len(p_set | h_set)
    jaccard = overlap / union if union > 0 else 0.0
    p_len, h_len = len(p), len(h)
    
    neg_p = sum(word in NEG_WORDS for word in p)
    neg_h = sum(word in NEG_WORDS for word in h)
    contrast_p = sum(word in CONTRAST_WORDS for word in p)
    contrast_h = sum(word in CONTRAST_WORDS for word in h)
    hyp_in_prem = int(" ".join(h) in " ".join(p))

    return [
        float(overlap), float(jaccard), float(p_len), float(h_len), 
        float(abs(p_len - h_len)), float(neg_p), float(neg_h), 
        float(abs(neg_p - neg_h)), float(contrast_p), float(contrast_h), float(hyp_in_prem)
    ]

MAX_LEN = 128

def preprocess(example):
    encoded = tokenizer(example["premise"], example["hypothesis"], truncation=True, padding="max_length", max_length=MAX_LEN)
    encoded["features"] = extract_features(example["premise"], example["hypothesis"])
    encoded["label"] = int(example["label"])
    return encoded

train_dataset = train_dataset.map(preprocess)
dev_dataset = dev_dataset.map(preprocess)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "features", "label"])
dev_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "features", "label"])

Map:   0%|          | 0/24432 [00:00<?, ? examples/s]

Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

## 3. Model Architecture
The `MultiExpertNLI` model uses three experts: Semantic (BERT), Lexical (Feature MLP), and Logic (Feature MLP).

In [6]:
class MultiExpertNLI(nn.Module):
    def __init__(self, model_name="bert-base-uncased", num_labels=2, feature_dim=11):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.semantic_classifier = nn.Sequential(nn.Linear(hidden_size, hidden_size), nn.ReLU(), nn.Dropout(0.1), nn.Linear(hidden_size, num_labels))
        self.lexical_expert = nn.Sequential(nn.Linear(feature_dim, 32), nn.ReLU(), nn.Dropout(0.1), nn.Linear(32, num_labels))
        self.logic_expert = nn.Sequential(nn.Linear(feature_dim, 32), nn.ReLU(), nn.Dropout(0.1), nn.Linear(32, num_labels))
        self.gating_network = nn.Sequential(nn.Linear(hidden_size + feature_dim, 64), nn.ReLU(), nn.Dropout(0.1), nn.Linear(64, 3))
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask, features, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        
        s_logits = self.semantic_classifier(cls)
        lex_logits = self.lexical_expert(features.float())
        log_logits = self.logic_expert(features.float())

        gate_input = torch.cat([cls, features.float()], dim=1)
        gate_weights = torch.softmax(self.gating_network(gate_input), dim=1)

        final_logits = (gate_weights[:, 0:1] * s_logits + 
                        gate_weights[:, 1:2] * lex_logits + 
                        gate_weights[:, 2:3] * log_logits)

        loss = self.loss_fn(final_logits, labels) if labels is not None else None
        return {"loss": loss, "logits": final_logits}

model = MultiExpertNLI()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 4. Training
Setting up the `Trainer` and starting the fine-tuning process.

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds), "macro_f1": f1_score(labels, preds, average="macro")}

training_args = TrainingArguments(
    output_dir="results_multi_expert",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    save_strategy="epoch",
    eval_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model, 
    args=training_args, 
    train_dataset=train_dataset, 
    eval_dataset=dev_dataset, 
    compute_metrics=compute_metrics
)

trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.431194,0.414601,0.812500,0.812498
2,0.285225,0.454063,0.829276,0.828404
3,0.148989,0.657340,0.828385,0.828163


RuntimeError: on_train_begin must be called before on_evaluate